# Tutorial 4: Developing a Custom Interface (Dev)

This tutorial shows how to create a custom data interface for scivianna, enabling visualization of any data source through the GUI.

## What is an Interface?

An **Interface** in scivianna is a bridge between your data and the visualization panels. It tells scivianna:
- How to load your data from files
- What fields are available for display
- How to compute 2D slices for geometry panels
- How to get values at specific points (optional)

## Interface Class Hierarchy

```
GenericInterface          - Base class: file I/O, serialization
    └── Geometry2D        - Adds 2D geometry slice computation
        ├── Geometry2DPolygon  - Returns polygon vertex lists
        └── Geometry2DGrid     - Returns rasterized grid data
ValueAtLocation           - Mixin: point value queries
Value1DAtLocation         - Mixin: 1D line value queries
OverLine                  - Mixin: line integration computation
```

**You compose interfaces by inheriting from multiple classes.** For example:
```python
class MyInterface(Geometry2DPolygon, ValueAtLocation):
    pass
```

## Required Methods by Base Class

| Base Class | Methods to Implement |
|-----------|---------------------|
| `GenericInterface` | `read_file()`, `get_labels()`, `get_label_coloring_mode()`, `get_file_input_list()` |
| `Geometry2D` (Polygon/Grid) | + `compute_2D_data()` |
| `ValueAtLocation` | + `get_value()` |
| `Value1DAtLocation` | + `get_1D_value()` |
| `OverLine` | + `compute_1D_line_data()` |

## Step 1: Minimal Interface (GenericInterface only)

The absolute minimum interface needs 4 methods. This won't show 2D graphics but will work with 1D panels.

In [ ]:
from scivianna.interface.generic_interface import GenericInterface
from scivianna.slave import ComputeSlave

class MinimalInterface(GenericInterface):
    """Simplest possible interface - reads numeric columns from a text file."""
    
    def __init__(self, context=None):
        self.data = {}       # {file_label: list of lists}
        self.labels = []     # Available field names
    
    def read_file(self, file_path: str, file_label: str) -> None:
        """Load a simple text/CSV file with numeric columns."""
        import csv
        rows = []
        with open(file_path, 'r') as f:
            reader = csv.reader(f)
            for row in reader:
                rows.append([float(x) for x in row if x.strip()])
        self.data[file_label] = rows
        self.labels.append(file_label)
        print(f"Loaded {file_path} -> {len(rows)} rows")
    
    def get_labels(self):
        """Return available field names."""
        return self.labels
    
    def get_label_coloring_mode(self, label: str):
        from scivianna.enums import VisualizationMode
        return VisualizationMode.FLOAT  # Data is numeric
    
    def get_file_input_list(self):
        """File filters shown in the GUI file picker."""
        return [("csv txt", "Text files (*.csv *.txt)")]

# Create a ComputeSlave with this interface
slave = ComputeSlave(MinimalInterface)
print(f"Available labels: {slave.get_labels()}")

## Step 2: Adding 2D Geometry (Geometry2DPolygon)

To display geometry in a 2D panel, inherit from `Geometry2DPolygon` and implement `compute_2D_data()`. This method receives viewport bounds and must return a `Data2D` object.

**Key concept**: `compute_2D_data()` is called whenever the panel needs to render. It receives:
- `u`, `v`: direction vectors of the 2D viewport
- `u_min`, `u_max`, `v_min`, `v_max`: viewport bounds
- `w_value`: the coordinate perpendicular to the 2D plane

**Return value**: A tuple `(Data2D, updated_bool)` where:
- `Data2D(vertices, indices, colors, data_type, mode)`
- `updated_bool`: True if geometry changed since last call

In [ ]:
import numpy as np
from scivianna.interface.generic_interface import Geometry2DPolygon
from scivianna.data.data2d import Data2D
from scivianna.enums import VisualizationMode, DataType, GeometryType
from scivianna.slave import ComputeSlave
import multiprocessing as mp
from typing import Tuple, Dict, Any

class CircleInterface(Geometry2DPolygon):
    """Displays a colored circle - minimal Geometry2DPolygon example."""
    
    # Required class attributes
    geometry_type = GeometryType.GEOMETRY_2D  # Shows 3D axis controls in GUI
    data_type = DataType.DATA_POLYGON          # Returns polygon vertex lists
    
    def __init__(self, context=None):
        self.polygons = {}    # {label: [vertices_array, ...]}
        self.colors = {}      # {label: color_values_array}
        self.labels = []
    
    def read_file(self, file_path: str, file_label: str) -> None:
        """For this example, we generate a circle programmatically."""
        angles = np.linspace(0, 2*np.pi, 36, endpoint=False)
        vertices = np.column_stack([np.cos(angles), np.sin(angles)])
        self.polygons[file_label] = [vertices]
        self.colors[file_label] = np.ones(36) * 0.5
        self.labels.append(file_label)
        print(f"Created circle geometry: {file_label}")
    
    def get_labels(self):
        return self.labels
    
    def get_label_coloring_mode(self, label: str):
        return VisualizationMode.FLOAT
    
    def get_file_input_list(self):
        return [("dat", "Data files (*.dat)")]
    
    def compute_2D_data(
        self,
        u: Tuple[float, float, float],
        v: Tuple[float, float, float],
        u_min: float, u_max: float,
        v_min: float, v_max: float,
        w_value: float,
        q_tasks: mp.Queue,
        options: Dict[str, Any],
        caller: str = "API",
    ):
        """Return polygon data for the 2D panel."""
        for label, vertices_list in self.polygons.items():
            vertices = vertices_list[0]
            colors = self.colors[label]
            indices = np.arange(len(vertices))
            
            data2d = Data2D(
                vertices,      # Nx2 array of polygon vertices
                indices,       # Vertex indices (identity for simple case)
                colors,        # Color values per vertex
                DataType.DATA_POLYGON,
                VisualizationMode.FLOAT
            )
            return data2d, True  # (data, was_updated)
        
        # No data available
        empty = Data2D(
            np.array([]), np.array([]), np.array([]),
            DataType.DATA_POLYGON, VisualizationMode.FLOAT
        )
        return empty, True

# Create slave
slave = ComputeSlave(CircleInterface)
# slave.read_file("dummy.dat", "circle")
# In the GUI, this would display as a colored polygon in Panel2D

## Step 3: Adding Point Value Queries (ValueAtLocation)

Inherit from `ValueAtLocation` to enable panels to query field values at specific (x, y, z) positions. This is used by tooltips, cursor tracking, and data probes.

**Required method**: `get_value(xyz: Tuple[float, float, float], label: str) -> Optional[float]`

In [ ]:
import numpy as np
from scivianna.interface.generic_interface import Geometry2DPolygon, ValueAtLocation
from scivianna.data.data2d import Data2D
from scivianna.enums import VisualizationMode, DataType, GeometryType
from typing import Tuple, Dict, Any, Optional
import multiprocessing as mp

class CircleWithValues(Geometry2DPolygon, ValueAtLocation):
    """Circle geometry with point value queries."""
    
    geometry_type = GeometryType.GEOMETRY_2D
    data_type = DataType.DATA_POLYGON
    
    def __init__(self, context=None):
        self.polygons = {}
        self.labels = []
        # Pre-compute vertex positions for value queries
        angles = np.linspace(0, 2*np.pi, 36, endpoint=False)
        self.vertex_positions = np.column_stack([np.cos(angles), np.sin(angles)])
    
    def read_file(self, file_path: str, file_label: str) -> None:
        vertices = self.vertex_positions.copy()
        self.polygons[file_label] = [vertices]
        self.labels.append(file_label)
        print(f"Created circle with value queries: {file_label}")
    
    def get_labels(self):
        return self.labels
    
    def get_label_coloring_mode(self, label: str):
        return VisualizationMode.FLOAT
    
    def get_file_input_list(self):
        return [("dat", "Data files (*.dat)")]
    
    def compute_2D_data(self, u, v, u_min, u_max, v_min, v_max, w_value, q_tasks, options, caller="API"):
        """Return polygon data."""
        for label, vertices_list in self.polygons.items():
            vertices = vertices_list[0]
            # Color by angle (theta)
            colors = np.arctan2(vertices[:,1], vertices[:,0]) / np.pi
            indices = np.arange(len(vertices))
            data2d = Data2D(
                vertices, indices, colors,
                DataType.DATA_POLYGON, VisualizationMode.FLOAT
            )
            return data2d, True
        empty = Data2D(np.array([]), np.array([]), np.array([]), DataType.DATA_POLYGON, VisualizationMode.FLOAT)
        return empty, True
    
    def get_value(self, xyz: Tuple[float, float, float], label: str) -> Optional[float]:
        """Return the value at a 3D point. For a circle in XY plane, return angle if on circle."""
        x, y, z = xyz
        # Check if point is approximately on the unit circle
        distance = np.sqrt(x**2 + y**2)
        if abs(distance - 1.0) < 0.1 and abs(z) < 0.1:
            return float(np.arctan2(y, x) / np.pi)  # Return normalized angle
        return None  # Point not in domain

# Test the value query
iface = CircleWithValues()
iface.read_file("dummy.dat", "circle")
print(f"Value at (1,0,0): {iface.get_value((1.0, 0.0, 0.0), 'circle')}")
print(f"Value at (0,1,0): {iface.get_value((0.0, 1.0, 0.0), 'circle')}")
print(f"Value at origin:   {iface.get_value((0.0, 0.0, 0.0), 'circle')}")

## Step 4: Grid-Based Geometry (Geometry2DGrid)

For raster/image data, inherit from `Geometry2DGrid` instead of `Geometry2DPolygon`. This returns a rectangular grid of values.

**Key difference**: Instead of vertex lists, you return a 2D array that gets displayed as an image/heatmap.

In [ ]:
import numpy as np
from scivianna.interface.generic_interface import Geometry2DGrid
from scivianna.data.data2d import Data2D
from scivianna.enums import VisualizationMode, DataType, GeometryType
from typing import Tuple, Dict, Any
import multiprocessing as mp

class HeatmapInterface(Geometry2DGrid):
    """Displays a 2D heatmap - minimal Geometry2DGrid example."""
    
    geometry_type = GeometryType.GEOMETRY_2D
    data_type = DataType.DATA_GRID
    
    def __init__(self, context=None):
        self.grids = {}   # {label: 2D array}
        self.bounds = {}  # {label: (x_min, x_max, y_min, y_max)}
        self.labels = []
    
    def read_file(self, file_path: str, file_label: str) -> None:
        """For this example, generate a heatmap pattern."""
        # Create a 2D Gaussian-like pattern
        x = np.linspace(-1, 1, 50)
        y = np.linspace(-1, 1, 50)
        X, Y = np.meshgrid(x, y)
        Z = np.exp(-(X**2 + Y**2) * 5)
        self.grids[file_label] = Z
        self.bounds[file_label] = (-1, 1, -1, 1)
        self.labels.append(file_label)
        print(f"Created heatmap: {file_label}")
    
    def get_labels(self):
        return self.labels
    
    def get_label_coloring_mode(self, label: str):
        return VisualizationMode.FLOAT
    
    def get_file_input_list(self):
        return [("dat", "Data files (*.dat)")]
    
    def compute_2D_data(self, u, v, u_min, u_max, v_min, v_max, w_value, q_tasks, options, caller="API"):
        """Return grid data for the 2D panel."""
        for label, grid_data in self.grids.items():
            x_min, x_max, y_min, y_max = self.bounds[label]
            # Create vertex grid matching the data array
            nx, ny = grid_data.shape
            xs = np.linspace(x_min, x_max, nx)
            ys = np.linspace(y_min, y_max, ny)
            X, Y = np.meshgrid(xs, ys)
            vertices = np.column_stack([X.ravel(), Y.ravel()])
            colors = grid_data.ravel()
            # Create triangle indices for the grid
            indices = []
            for i in range(nx - 1):
                for j in range(ny - 1):
                    idx = i * ny + j
                    indices.extend([idx, idx + 1, idx + ny])
                    indices.extend([idx + 1, idx + ny + 1, idx + ny])
            indices = np.array(indices)
            
            data2d = Data2D(
                vertices, indices, colors,
                DataType.DATA_GRID, VisualizationMode.FLOAT
            )
            return data2d, True
        
        empty = Data2D(np.array([]), np.array([]), np.array([]), DataType.DATA_GRID, VisualizationMode.FLOAT)
        return empty, True

# Create interface instance
iface = HeatmapInterface()
iface.read_file("dummy.dat", "heatmap")
print(f"Grid shape: {iface.grids['heatmap'].shape}")

## Step 5: Serialization (to_json / from_json)

Override `to_json()` and `from_json()` to save/restore interface state. This is essential for:
- Saving/loading user configurations
- Sharing visualization setups between sessions

**Required class attribute**: `is_serializable = True`

In [ ]:
import numpy as np
from scivianna.interface.generic_interface import Geometry2DPolygon
from scivianna.data.data2d import Data2D
from scivianna.enums import VisualizationMode, DataType, GeometryType
from typing import Tuple, Dict, Any
import multiprocessing as mp

class SerializableInterface(Geometry2DPolygon):
    """Example of a serializable interface."""
    
    is_serializable = True  # Enable serialization
    geometry_type = GeometryType.GEOMETRY_2D
    data_type = DataType.DATA_POLYGON
    
    def __init__(self, context=None):
        self.polygons = {}
        self.labels = []
        # Custom parameters that should be saved
        self.radius = 1.0
        self.n_vertices = 36
    
    def read_file(self, file_path: str, file_label: str) -> None:
        angles = np.linspace(0, 2*np.pi, self.n_vertices, endpoint=False)
        vertices = self.radius * np.column_stack([np.cos(angles), np.sin(angles)])
        self.polygons[file_label] = [vertices]
        self.labels.append(file_label)
    
    def get_labels(self):
        return self.labels
    
    def get_label_coloring_mode(self, label: str):
        return VisualizationMode.FLOAT
    
    def get_file_input_list(self):
        return [("dat", "Data files (*.dat)")]
    
    def compute_2D_data(self, u, v, u_min, u_max, v_min, v_max, w_value, q_tasks, options, caller="API"):
        for label, vertices_list in self.polygons.items():
            vertices = vertices_list[0]
            colors = np.ones(len(vertices)) * 0.5
            indices = np.arange(len(vertices))
            return Data2D(vertices, indices, colors, DataType.DATA_POLYGON, VisualizationMode.FLOAT), True
        empty = Data2D(np.array([]), np.array([]), np.array([]), DataType.DATA_POLYGON, VisualizationMode.FLOAT)
        return empty, True
    
    def to_json(self) -> dict:
        """Save custom parameters."""
        return {
            "radius": self.radius,
            "n_vertices": self.n_vertices,
            "polygons_keys": list(self.polygons.keys()),
            "labels": self.labels
        }
    
    @classmethod
    def from_json(cls, info_dict: dict):
        """Recreate interface from saved state."""
        instance = cls()
        instance.radius = info_dict.get("radius", 1.0)
        instance.n_vertices = info_dict.get("n_vertices", 36)
        instance.labels = info_dict.get("labels", [])
        # Note: actual polygon data would need to be regenerated
        # since we only saved parameters, not the full arrays
        return instance

# Test serialization
iface1 = SerializableInterface()
iface1.radius = 2.0
iface1.n_vertices = 72
saved = iface1.to_json()
print(f"Saved state: {saved}")

iface2 = SerializableInterface.from_json(saved)
print(f"Restored radius: {iface2.radius}, n_vertices: {iface2.n_vertices}")

## Step 6: Complete Example - Multi-Field Polygon Interface

A realistic interface that:
- Reads polygon data from files (one polygon per file)
- Supports multiple scalar fields per polygon
- Provides point value queries via nearest-neighbor interpolation
- Is serializable

In [ ]:
import numpy as np
from scivianna.interface.generic_interface import Geometry2DPolygon, ValueAtLocation
from scivianna.data.data2d import Data2D
from scivianna.enums import VisualizationMode, DataType, GeometryType
from typing import Tuple, Dict, Any, Optional, List
import multiprocessing as mp

class MultiFieldPolygonInterface(Geometry2DPolygon, ValueAtLocation):
    """
    Complete interface for polygon data with multiple scalar fields.
    
    File format (one polygon per file):
        # Comment lines start with #
        x1 y1 field1_value field2_value ...
        x2 y2 field1_value field2_value ...
        ...
    """
    
    is_serializable = True
    geometry_type = GeometryType.GEOMETRY_2D
    data_type = DataType.DATA_POLYGON
    
    def __init__(self, context=None):
        self.polygons: Dict[str, np.ndarray] = {}       # {label: vertices Nx2}
        self.fields: Dict[str, Dict[str, np.ndarray]] = {}  # {label: {field_name: values N}}
        self.available_fields: List[str] = []            # Field names
        self.labels: List[str] = []                      # File labels
        self.current_field: str = None                   # Currently selected field
    
    def read_file(self, file_path: str, file_label: str) -> None:
        """Read a polygon file with optional scalar fields."""
        rows = []
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                rows.append([float(x) for x in line.split()])
        
        data = np.array(rows)
        vertices = data[:, :2]  # First 2 columns are x, y
        field_values = data[:, 2:] if data.shape[1] > 2 else np.ones((len(vertices), 1)) * 0.5
        
        self.polygons[file_label] = vertices
        self.labels.append(file_label)
        
        # Store field values (column 0 = primary field)
        if file_label not in self.fields:
            self.fields[file_label] = {}
        self.fields[file_label]["primary"] = field_values[:, 0]
        
        # Store additional fields if present
        for i in range(1, field_values.shape[1]):
            self.fields[file_label][f"field_{i}"] = field_values[:, i]
            if f"field_{i}" not in self.available_fields:
                self.available_fields.append(f"field_{i}")
        
        if "primary" not in self.available_fields:
            self.available_fields.insert(0, "primary")
        
        self.current_field = "primary"
        print(f"Loaded: {file_label} ({len(vertices)} vertices)")
    
    def get_labels(self) -> List[str]:
        return self.labels
    
    def get_label_coloring_mode(self, label: str) -> VisualizationMode:
        return VisualizationMode.FLOAT
    
    def get_file_input_list(self) -> List[Tuple[str, str]]:
        return [("dat txt", "Data files (*.dat *.txt)")]
    
    def compute_2D_data(self, u, v, u_min, u_max, v_min, v_max, w_value, q_tasks, options, caller="API"):
        """Return polygon data colored by the selected field."""
        for label, vertices in self.polygons.items():
            if label not in self.fields or self.current_field is None:
                continue
            
            colors = self.fields[label].get(self.current_field, np.ones(len(vertices)))
            indices = np.arange(len(vertices))
            
            data2d = Data2D(
                vertices.copy(),
                indices,
                colors.copy(),
                DataType.DATA_POLYGON,
                VisualizationMode.FLOAT
            )
            return data2d, True
        
        empty = Data2D(np.array([]), np.array([]), np.array([]), DataType.DATA_POLYGON, VisualizationMode.FLOAT)
        return empty, True
    
    def get_value(self, xyz: Tuple[float, float, float], label: str) -> Optional[float]:
        """Return value at a point using nearest-neighbor interpolation."""
        if label not in self.polygons or self.current_field is None:
            return None
        
        vertices = self.polygons[label]
        colors = self.fields[label].get(self.current_field, np.ones(len(vertices)))
        
        # Find nearest vertex
        dx = vertices[:, 0] - xyz[0]
        dy = vertices[:, 1] - xyz[1]
        dists = dx**2 + dy**2
        nearest_idx = np.argmin(dists)
        
        if dists[nearest_idx] < 0.01:  # Within threshold
            return float(colors[nearest_idx])
        return None
    
    def set_field(self, field_name: str):
        """Switch the currently displayed field."""
        if field_name in self.available_fields:
            self.current_field = field_name
            print(f"Switched to field: {field_name}")
    
    def get_available_fields(self) -> List[str]:
        """Return list of available fields for the current label."""
        return self.available_fields
    
    def to_json(self) -> dict:
        return {
            "labels": self.labels,
            "current_field": self.current_field,
            "available_fields": self.available_fields
        }
    
    @classmethod
    def from_json(cls, info_dict: dict):
        instance = cls()
        instance.labels = info_dict.get("labels", [])
        instance.current_field = info_dict.get("current_field")
        instance.available_fields = info_dict.get("available_fields", [])
        return instance

# Test the complete interface
iface = MultiFieldPolygonInterface()

# Simulate reading a file by directly setting data
angles = np.linspace(0, 2*np.pi, 20, endpoint=False)
x = np.cos(angles)
y = np.sin(angles)
field1 = np.sin(angles)  # Varies with angle
field2 = np.cos(2*angles)  # Different pattern

# Create mock file content
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.dat', delete=False) as f:
    for i in range(len(angles)):
        f.write(f"{x[i]:.4f} {y[i]:.4f} {field1[i]:.4f} {field2[i]:.4f}\n")
    temp_path = f.name

iface.read_file(temp_path, "test_polygon")
print(f"Labels: {iface.get_labels()}")
print(f"Available fields: {iface.get_available_fields()}")

# Test field switching
iface.set_field("field_1")
iface.set_field("primary")

# Test value query
val = iface.get_value((0.9, 0.1, 0.0), "test_polygon")
print(f"Value at (0.9, 0.1, 0): {val}")

# Test serialization
saved = iface.to_json()
restored = MultiFieldPolygonInterface.from_json(saved)
print(f"Restored labels: {restored.labels}, field: {restored.current_field}")

os.unlink(temp_path)

## Summary: Interface Development Checklist

1. **Choose base classes**: Start with `GenericInterface`, then add mixins as needed:
   - `Geometry2DPolygon` for polygon/line geometry
   - `Geometry2DGrid` for raster/grid data
   - `ValueAtLocation` for point queries
   - `Value1DAtLocation` for line queries
   - `OverLine` for line integration

2. **Implement required methods** based on base classes (see table at top)

3. **Set class attributes**: `geometry_type`, `data_type`, and optionally `is_serializable`

4. **Handle the compute_2D_data signature carefully** - it always receives these parameters:
   ```python
   def compute_2D_data(self, u, v, u_min, u_max, v_min, v_max, w_value, q_tasks, options, caller="API"):
   ```

5. **Return correct Data2D format** matching your `data_type`:
   - `DATA_POLYGON`: vertices as vertex lists per polygon
   - `DATA_GRID`: vertices and indices for a triangular grid

6. **Test with the ComputeSlave** before integrating into the GUI